# RAG Question-Answering Pipeline

**Local VS Code Edition — ChromaDB + Vertex AI Embeddings + Gemini 2.5 Pro**

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline that
answers questions grounded in 100 peer-reviewed Epidemiology research papers, stored in Google Cloud Storage 

## Architecture

| Stage | Component | Purpose |
|-------|-----------|---------|
| Document source | Google Cloud Storage (GCS) | Holds the source PDFs |
| Loader | `PyPDFLoader` (LangChain) | Extracts page-level text from each PDF |
| Splitter | `RecursiveCharacterTextSplitter` | Chunks text into overlapping windows |
| Embeddings | Vertex AI `text-embedding-005` | Encodes chunks into 768-dim vectors |
| Vector store | **ChromaDB (local, persistent)** | Indexes vectors for similarity search |
| Retriever | LangChain similarity retriever | Top-k chunk lookup for a query |
| LLM | **Gemini 2.5 Pro** via `ChatGoogleGenerativeAI` | Generates the grounded answer |
| Orchestration | LangChain retrieval chain | Stitches retriever + LLM with a strict prompt |

## Notebook structure

1. **Installs** — single consolidated cell
2. **Imports** — single consolidated cell
3. **Configuration** — project, bucket, paths, hyperparameters
4. **Phase 1** — Vertex AI client initialization
5. **Phase 2** — Document download from GCS, parsing, and chunking
6. **Phase 5** — ChromaDB vector store creation
7. **Phase 6** — Batched embedding and ingestion (with retry on quota errors)
8. **Phase 7** — Sanity-check similarity search
9. **Phase 8** — LLM, retriever, prompt template, RAG chain, and `ask()` function
10. **Phase 9** — End-to-end question-answering examples




## Authentication (One-Time Setup)

Before launching the notebook, run the following **once** in your terminal:

```bash
# Authenticate your user identity (opens a browser)
gcloud auth application-default login

# Pin the active project so client libraries use it by default
gcloud config set project gill-adta5770-rag
```

This writes Application Default Credentials (ADC) to
`~/.config/gcloud/application_default_credentials.json` (Linux/macOS) or
`%APPDATA%\gcloud\application_default_credentials.json` (Windows). Every
Google client library used in this notebook — `google.cloud.storage`,
`google.cloud.aiplatform`, `vertexai`, `langchain_google_vertexai`,
`langchain_google_genai` — will pick up those credentials automatically.

**Required IAM roles** on your user account for the target project:

- `roles/storage.objectViewer` — to read the source PDFs from the GCS bucket
- `roles/aiplatform.user` — to call Vertex AI embeddings and Gemini

If you ever see `google.auth.exceptions.DefaultCredentialsError`, re-run
`gcloud auth application-default login`.


## 1. Installs

A single consolidated install cell. Run this once per environment, then
restart the Jupyter kernel before continuing.


In [1]:
# Consolidated dependency installation

# Google Cloud SDKs (auth + GCS + Vertex AI + Gemini)
%pip install --quiet --upgrade google-cloud-aiplatform google-cloud-storage vertexai google-genai

# LangChain core stack
%pip install --quiet --upgrade langchain langchain-core langchain-classic langchain-community langchain-text-splitters

# LangChain Google integrations (Vertex AI embeddings, Gemini chat model)
%pip install --quiet --upgrade langchain-google-community langchain-google-vertexai langchain-google-genai

# Local vector store (replaces Vertex AI Matching Engine)
%pip install --quiet --upgrade chromadb langchain-chroma

# PDF parsing (PyPDFLoader backend — no OCR / poppler / tesseract needed
%pip install --quiet --upgrade pypdf

print("Installation complete. Please restart the kernel before running the next cell.")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Installation complete. Please restart the kernel before running the next cell.


## 2. Imports

All imports are consolidated into this single cell so that dependency usage is
transparent at a glance and ordering issues cannot arise between phases.


In [1]:
# --- Standard library ---
import os
import time
import textwrap

# --- Google Cloud / Vertex AI ---
from google.cloud import storage                          # GCS bucket access
from google.cloud import aiplatform                       # Vertex AI SDK (init only)
import vertexai                                           # Vertex AI SDK (init only)
from google.api_core.exceptions import ResourceExhausted  # For quota / rate-limit retry

# --- LangChain core (prompts, debug toggle) ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.globals import set_verbose

# --- LangChain document loading and chunking ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- LangChain RAG chain primitives ---
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# --- Vertex AI embedding model wrapper ---
from langchain_google_vertexai import VertexAIEmbeddings

# --- Gemini chat model wrapper ---
from langchain_google_genai import ChatGoogleGenerativeAI

# --- ChromaDB vector store (local, persistent — replaces Vertex AI Matching Engine) ---
from langchain_chroma import Chroma

# Quick version sanity check — useful for reproducibility in the report.
import langchain
from langchain_core import __version__ as langchain_core_version
from langchain_classic import __version__ as langchain_classic_version
from langchain_community import __version__ as langchain_community_version
print(f"aiplatform SDK version       : {aiplatform.__version__}")
print(f"Vertex AI SDK version        : {vertexai.__version__}")
print(f"LangChain version            : {langchain.__version__}")
print(f"LangChain Core version       : {langchain_core_version}")
print(f"LangChain Classic version    : {langchain_classic_version}")
print(f"LangChain Community version  : {langchain_community_version}")


C:\Users\agill\AppData\Local\Temp\ipykernel_5160\2584870985.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


aiplatform SDK version       : 1.156.0
Vertex AI SDK version        : 1.156.0
LangChain version            : 1.3.4
LangChain Core version       : 1.4.1
LangChain Classic version    : 1.0.7
LangChain Community version  : 0.4.2


## 3. Configuration

All tunable parameters live here so the rest of the notebook can be re-read
without scrolling back for project IDs, paths, or hyperparameters.


In [10]:
# --- Google Cloud project + region ---
PROJECT_ID = 'gill-adta-5770-rag'   # GCP project hosting the bucket and Vertex AI quotas
REGION     = 'us-central1'         # Region for Vertex AI (must support text-embedding-005 and Gemini)

# --- Source PDF location in Google Cloud Storage ---
BUCKET_NAME   = 'domain-docs'         # GCS bucket containing the corpus
folder_prefix = 'documents/pdfs/'     # Folder (object key prefix) inside the bucket
BUCKET_URI    = f"gs://{BUCKET_NAME}/{folder_prefix}"  # Full gs:// URI for logging only

# --- Local working directories (created at runtime if they don't exist) ---
LOCAL_PDF_DIR        = './temp_pdfs'    # Where GCS PDFs are downloaded for parsing
CHROMA_PERSIST_DIR   = './chroma_db'    # Where ChromaDB stores its on-disk index
CHROMA_COLLECTION    = 'rag_qa_collection'  # Logical name for the Chroma collection

# --- Embedding model (Vertex AI) ---
EMBEDDING_MODEL_NAME = 'text-embedding-005'  # 768-dim Google text embedding model
DIMENSIONS           = 768                   # Output dimensionality of text-embedding-005

# --- Retrieval hyperparameters ---
NUMBER_OF_RESULTS         = 10   # Top-k chunks to retrieve for each query
SEARCH_DISTANCE_THRESHOLD = 0.6  # Reserved for future similarity-score filtering

# --- Ingestion hyperparameters ---
CHUNK_SIZE     = 400   # Characters per chunk (small chunks = more precise retrieval)
CHUNK_OVERLAP  = 50    # Overlap between adjacent chunks (preserves boundary context)
BATCH_SIZE     = 25    # Chunks per embedding API call
MAX_RETRIES    = 5     # Retries on Vertex AI quota errors (exponential backoff)

# --- LLM (generation) ---
LLM_MODEL_NAME = 'gemini-2.5-pro'  # Reasoning model used to compose the final answer

print(f"Project       : {PROJECT_ID}")
print(f"Region        : {REGION}")
print(f"Source bucket : {BUCKET_URI}")
print(f"Chroma store  : {CHROMA_PERSIST_DIR}")


Project       : gill-adta-5770-rag
Region        : us-central1
Source bucket : gs://domain-docs/documents/pdfs/
Chroma store  : ./chroma_db


## Phase 1 — Vertex AI Client Initialization

Although the **vector store** runs locally (ChromaDB), we still call Vertex AI
for two things:

1. **Embedding generation** via `text-embedding-005`
2. **Answer generation** via Gemini 2.5 Pro

Both client SDKs (`aiplatform` and `vertexai`) need to be initialized once
with the project ID and region so subsequent calls know where to route
requests. We also instantiate a `storage.Client` to verify that
Application Default Credentials are working before we hit the network in
Phase 2.




In [11]:
# Initialize the Vertex AI SDKs
aiplatform.init(project=PROJECT_ID, location=REGION)
vertexai.init(project=PROJECT_ID, location=REGION)

# Instantiate a GCS client
storage_client = storage.Client(project=PROJECT_ID)
print(f"Authenticated with project: {storage_client.project}")


Authenticated with project: gill-adta-5770-rag


## Phase 2 — Document Loading & Chunking

This phase performs three things:

1. **List** the PDFs that live under `gs://{BUCKET_NAME}/{folder_prefix}`.
2. **Download** each PDF locally into `./temp_pdfs/` and parse it with
   `PyPDFLoader`. We download first because parsing a local file is
   dramatically faster than streaming through `GCSFileLoader`, and it
   avoids invoking the heavy `unstructured` / OCR path.
3. **Chunk** every page into overlapping windows using
   `RecursiveCharacterTextSplitter`. Each chunk inherits page metadata and
   is tagged with its source path, document filename, and a sequential
   chunk index. These tags are later used for retrieval filtering and
   citation display in the `ask()` output.


### 2a. List the source PDFs

In [12]:
# Print every blob under the configured prefix so it's clear which files
# will be processed downstream. This is a read-only listing call.
bucket = storage_client.bucket(BUCKET_NAME)
print(f"Listing PDFs in {BUCKET_URI}\n")

for blob in bucket.list_blobs(prefix=folder_prefix):
    print(f"  - {blob.name}")


Listing PDFs in gs://domain-docs/documents/pdfs/

  - documents/pdfs/
  - documents/pdfs/10.1371_journal.pgph.0004406.pdf
  - documents/pdfs/10.1371_journal.pgph.0004688.pdf
  - documents/pdfs/10.1371_journal.pgph.0005862.pdf
  - documents/pdfs/10.1371_journal.pgph.0005880.pdf
  - documents/pdfs/10.1371_journal.pmed.1004635.pdf
  - documents/pdfs/10.1371_journal.pmed.1004893.pdf
  - documents/pdfs/10.1371_journal.pntd.0013953.pdf
  - documents/pdfs/10.1371_journal.pone.0326407.pdf
  - documents/pdfs/10.1371_journal.pone.0332787.pdf
  - documents/pdfs/10.1371_journal.pone.0333733.pdf
  - documents/pdfs/10.1371_journal.pone.0339125.pdf
  - documents/pdfs/10.1371_journal.pone.0339408.pdf
  - documents/pdfs/10.1371_journal.pone.0340084.pdf
  - documents/pdfs/10.1371_journal.pone.0340666.pdf
  - documents/pdfs/10.1371_journal.pone.0340696.pdf
  - documents/pdfs/10.1371_journal.pone.0341269.pdf
  - documents/pdfs/10.1371_journal.pone.0341603.pdf
  - documents/pdfs/10.1371_journal.pone.034160

### 2b. Download and parse each PDF

In [15]:
# Make sure the local download folder exists.
os.makedirs(LOCAL_PDF_DIR, exist_ok=True)

print(f"Processing documents from {BUCKET_URI}\n")

# Accumulator for parsed page-level Documents from every PDF.
all_documents = []

# Stream blobs under the prefix. Each blob is one object in GCS.
for blob in bucket.list_blobs(prefix=folder_prefix):
    # Skip "directory" placeholder blobs and non-PDF objects.
    if blob.name.endswith("/") or not blob.name.lower().endswith(".pdf"):
        continue

    print(f"  Loading document: {blob.name}")

    # 1. Download the PDF locally first (fast — avoids the heavy OCR path
    #    that GCSFileLoader triggers via the `unstructured` library).
    document_name = blob.name.split("/")[-1]
    local_file_path = os.path.join(LOCAL_PDF_DIR, document_name)
    blob.download_to_filename(local_file_path)

    # 2. Parse with PyPDFLoader — produces one LangChain Document per page.
    loader = PyPDFLoader(local_file_path)
    try:
        documents_from_blob = loader.load()
    except Exception as e:
        print(f"  [!] Skipping {document_name}: {e}")
        continue

    # 3. Stamp every page-document with provenance metadata. These fields
    #    are surfaced in the citation block of the `ask()` formatter and
    #    are also used for filtered retrieval in Phase 9.
    doc_source_prefix = f"gs://{BUCKET_NAME}"
    doc_source_suffix = "/".join(blob.name.split("/")[0:-1])
    source = f"{doc_source_prefix}/{doc_source_suffix}"

    for document in documents_from_blob:
        document.metadata["source"] = source                # gs://bucket/prefix
        document.metadata["document_name"] = document_name  # original PDF filename
        all_documents.append(document)

print(f"\n# of document pages loaded (pre-chunking) = {len(all_documents)}")


Processing documents from gs://domain-docs/documents/pdfs/

  Loading document: documents/pdfs/10.1371_journal.pgph.0004406.pdf
  Loading document: documents/pdfs/10.1371_journal.pgph.0004688.pdf
  Loading document: documents/pdfs/10.1371_journal.pgph.0005862.pdf
  Loading document: documents/pdfs/10.1371_journal.pgph.0005880.pdf
  Loading document: documents/pdfs/10.1371_journal.pmed.1004635.pdf
  Loading document: documents/pdfs/10.1371_journal.pmed.1004893.pdf
  Loading document: documents/pdfs/10.1371_journal.pntd.0013953.pdf
  Loading document: documents/pdfs/10.1371_journal.pone.0326407.pdf
  Loading document: documents/pdfs/10.1371_journal.pone.0332787.pdf
  Loading document: documents/pdfs/10.1371_journal.pone.0333733.pdf
  Loading document: documents/pdfs/10.1371_journal.pone.0339125.pdf
  Loading document: documents/pdfs/10.1371_journal.pone.0339408.pdf
  Loading document: documents/pdfs/10.1371_journal.pone.0340084.pdf
  Loading document: documents/pdfs/10.1371_journal.pone.

Ignoring wrong pointing object 67 0 (offset 0)


  Loading document: documents/pdfs/Controlling Bias in Clinical Research.pdf
  Loading document: documents/pdfs/Decentralized pandemic response and health equity - an analysis of socioeconomic disparities in COVID-19 mortality in Japan.pdf
  Loading document: documents/pdfs/Design and Analysis of Case-Control Studies.pdf
  Loading document: documents/pdfs/Disease Outbreak Detection and Forecasting A Review of Methods and Data Sources.pdf
  Loading document: documents/pdfs/ESPAUR report 2024 to 2025 - Antimicrobial utilisation and resistance.pdf
  Loading document: documents/pdfs/Ebola Virus Disease External Situation Report (DRC).pdf
  Loading document: documents/pdfs/Effectiveness of the Influenza Vaccine During the 2024-2025 Season.pdf
  Loading document: documents/pdfs/Epidemic Models for COVID-19 During the First Wave Methodological Review.pdf
  Loading document: documents/pdfs/Epidemiology of non-communicable diseases among professional drivers in LMICs- a systematic review and me

### 2c. Split pages into overlapping chunks

Smaller chunks (400 chars with 50-char overlap) give the retriever more
fine-grained units to match against. The overlap prevents key facts from
being cut at chunk boundaries.


In [16]:
# RecursiveCharacterTextSplitter walks down a list of separators
# (\n\n, \n, " ", "") until each piece fits within `chunk_size`.
# This produces semantically cleaner chunks than naive fixed-size slicing.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    is_separator_regex=False,
)

# Apply the splitter to the page-level Documents from Phase 2b.
doc_splits = text_splitter.split_documents(all_documents)

# Tag each chunk with a stable sequential index. Useful for debugging
# retrieval results and for downstream citation rendering.
for idx, split in enumerate(doc_splits):
    split.metadata["chunk"] = idx

print(f"# of chunks (post-splitting) = {len(doc_splits)}")


# of chunks (post-splitting) = 31825


## Phase 5 — Create the Local ChromaDB Vector Store

ChromaDB is an open-source vector database that runs entirely in-process
and persists to disk under `CHROMA_PERSIST_DIR`. We pair it with the same
Vertex AI embedding model the original notebook used (`text-embedding-005`)
so that the **embedding semantics are identical** to the cloud version —
only the storage and similarity-search backend has changed.

If `CHROMA_PERSIST_DIR` already exists and contains a previous run's
collection, Chroma will reopen it; otherwise it will create a new one.


In [17]:
# Vertex AI embedding model. Authenticated via Application Default Credentials.
# Output is a 768-dim float vector per input string.
embedding_model = VertexAIEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    project=PROJECT_ID,
)

# Open (or create) a persistent local Chroma collection. The `embedding_function`
# is invoked automatically on every `add_texts` and on every query — we never
# have to pass raw vectors ourselves.
vsvectordb = Chroma(
    collection_name=CHROMA_COLLECTION,
    embedding_function=embedding_model,
    persist_directory=CHROMA_PERSIST_DIR,
)

print(f"ChromaDB collection '{CHROMA_COLLECTION}' ready at: {CHROMA_PERSIST_DIR}")
print(f"Existing items in collection: {vsvectordb._collection.count()}")


C:\Users\agill\AppData\Local\Temp\ipykernel_5160\19037370.py:3: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding_model = VertexAIEmbeddings(
C:\Users\agill\AppData\Local\Temp\ipykernel_5160\19037370.py:3: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embedding_model = VertexAIEmbeddings(


ChromaDB collection 'rag_qa_collection' ready at: ./chroma_db
Existing items in collection: 0


## Phase 6 — Embed and Ingest Chunks

Each chunk is sent to Vertex AI for embedding, then written to ChromaDB
along with its metadata. We batch requests (`BATCH_SIZE = 25`) to stay
under the embedding API's tokens-per-minute quota, and we wrap each call in
an exponential-backoff retry loop so a transient `ResourceExhausted` does
not abort the whole ingest.

Re-running this cell will append duplicates. If you want a clean slate,
delete the `./chroma_db` directory and re-run Phase 5.


In [18]:
# Pull texts and metadatas out of the Document objects so we can batch them.
texts = [doc.page_content for doc in doc_splits]
metadatas = [doc.metadata for doc in doc_splits]

total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

# Iterate over fixed-size slices to keep request size bounded.
for i in range(0, len(texts), BATCH_SIZE):
    batch_texts     = texts[i : i + BATCH_SIZE]
    batch_metadatas = metadatas[i : i + BATCH_SIZE]
    batch_num       = i // BATCH_SIZE + 1

    print(f"Adding batch {batch_num}/{total_batches} (size: {len(batch_texts)})...")

    retries = 0
    while retries < MAX_RETRIES:
        try:
            # Embeds the batch via Vertex AI AND inserts into Chroma in one call.
            vsvectordb.add_texts(texts=batch_texts, metadatas=batch_metadatas)
            break  # Success — exit the retry loop.

        except ResourceExhausted:
            # Quota / rate-limit error. Back off exponentially (10s, 20s, 40s, ...).
            retries += 1
            sleep_time = (2 ** retries) * 5
            print(f"   [!] Quota hit. Sleeping {sleep_time}s before retry "
                  f"(attempt {retries}/{MAX_RETRIES})...")
            time.sleep(sleep_time)

        except Exception as e:
            # Anything else: log and bail on this batch so a single bad chunk
            # cannot freeze the whole ingest.
            print(f"   [!] Unexpected error on batch {batch_num}: {e}")
            break

    # Light pacing between batches to smooth the request rate.
    time.sleep(2)

print(f"\nIngestion complete. Collection now has "
      f"{vsvectordb._collection.count()} items.")


Adding batch 1/1273 (size: 25)...
Adding batch 2/1273 (size: 25)...
Adding batch 3/1273 (size: 25)...
Adding batch 4/1273 (size: 25)...
Adding batch 5/1273 (size: 25)...
Adding batch 6/1273 (size: 25)...
Adding batch 7/1273 (size: 25)...
Adding batch 8/1273 (size: 25)...
Adding batch 9/1273 (size: 25)...
Adding batch 10/1273 (size: 25)...
Adding batch 11/1273 (size: 25)...
Adding batch 12/1273 (size: 25)...
Adding batch 13/1273 (size: 25)...
Adding batch 14/1273 (size: 25)...
Adding batch 15/1273 (size: 25)...
Adding batch 16/1273 (size: 25)...
Adding batch 17/1273 (size: 25)...
Adding batch 18/1273 (size: 25)...
Adding batch 19/1273 (size: 25)...
Adding batch 20/1273 (size: 25)...
Adding batch 21/1273 (size: 25)...
Adding batch 22/1273 (size: 25)...
Adding batch 23/1273 (size: 25)...
Adding batch 24/1273 (size: 25)...
Adding batch 25/1273 (size: 25)...
Adding batch 26/1273 (size: 25)...
Adding batch 27/1273 (size: 25)...
Adding batch 28/1273 (size: 25)...
Adding batch 29/1273 (size: 2

## Phase 7 — Sanity-Check the Vector Store

Before wiring up the LLM, confirm that semantic search is returning
reasonable matches for a known concept from the corpus. If this returns
empty results or unrelated chunks, debug Phase 6 before moving on.


In [19]:
# k=2 keeps the output short for inspection; we'll bump k for real queries.
sanity_results = vsvectordb.similarity_search("What is epidemiology?", k=2)

for i, hit in enumerate(sanity_results, start=1):
    print(f"--- Hit {i} ---")
    print(f"document_name: {hit.metadata.get('document_name')}")
    print(f"chunk        : {hit.metadata.get('chunk')}")
    print(f"content      : {hit.page_content[:300]}...")
    print()


--- Hit 1 ---
document_name: introduction-to-epidemiology.pdf
chunk        : 31599
content      : Adapted from: Last JM, ed. A dictionary of epidemiology. 2nd ed. Toronto, Canada: Oxford University Press; 1988. 
Epidemiology — Defined 
Study of the distribution and 
determinants of health-related 
states among specified 
populations and the application 
of that study to the control of 
health pr...

--- Hit 2 ---
document_name: Infectious-disease-epidemiology.pdf
chunk        : 25360
content      : demos – people 
 logy – study  
 
Epidemiology is, thus, the study of what is upon the people.  In modern terms, it is the 
science of the distribution of disease and its determinants (causes).   
 
Epidemiology is also a process that uses the facts at hand as clues to point to new 
knowledge and so...



## Phase 8 — Build the RAG Chain and the `ask()` Function

This is the heart of the notebook. We assemble:

1. **The LLM** — Gemini 2.5 Pro with conservative decoding parameters
   (`temperature=0.2`) for factual answers.
2. **The retriever** — a similarity retriever over the Chroma collection,
   returning the top `NUMBER_OF_RESULTS` chunks.
3. **The prompt template** — instructs the model to answer **only** from
   the retrieved context and to refuse otherwise.
4. **The retrieval chain** — LangChain's `create_retrieval_chain` glues
   the retriever and the document-stuffing prompt together.
5. **`ask()`** — a thin wrapper that lets us pass per-query overrides
   (`k`, `filters`) and pretty-prints the citations alongside the answer.


### 8a. Instantiate the Gemini LLM

In [20]:
# Gemini 2.5 Pro via the LangChain wrapper. The wrapper authenticates
# through Application Default Credentials (the same `gcloud auth ADC` you ran
# at the start), and routes through the configured project + region.
llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL_NAME,
    project=PROJECT_ID,
    location=REGION,
    max_output_tokens=8192,  # Plenty of room for grounded answers + reasoning
    temperature=0.2,         # Low temperature → more deterministic, factual replies
    top_p=0.8,
    top_k=40,
    verbose=True,
)


### 8b. Configure the retriever

In [21]:
# `as_retriever` exposes the Chroma collection through the standard LangChain
# retriever interface. We start with no filter; `ask()` will inject one
# at query time when the caller supplies `filters=...`.
retriever = vsvectordb.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": NUMBER_OF_RESULTS,
    },
)


### 8c. Define the answer prompt

The prompt explicitly forbids the model from answering outside the
retrieved context. This is what turns a generic chat model into a
grounded, citation-anchored QA system.


In [22]:
# Strict prompt: the model must say "I cannot determine..." rather than
# inventing facts. The {input} and {context} placeholders are filled in by
# the retrieval chain at invocation time.
template = """SYSTEM: You are an intelligent assistant helping the users with their questions on research papers.

Question: {input}

Strictly use ONLY the following pieces of context to answer the question at the end. Think step-by-step and then answer.

Do not try to make up an answer:
  - If the answer to the question cannot be determined from the context alone, say "I cannot determine the answer to that."
  - If the context is empty, just say "I do not know the answer to that."

=============
{context}
=============

Question: {input}
Helpful Answer:"""

qa_prompt = ChatPromptTemplate.from_template(template)


### 8d. Wire the chain together

In [23]:
# `create_stuff_documents_chain` builds a chain that "stuffs" all retrieved
# documents into the {context} slot of the prompt and asks the LLM for an answer.
combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt)

# `create_retrieval_chain` then prepends the retriever step so that calling
# `.invoke({"input": question})` will: (1) embed the query, (2) fetch top-k
# chunks from Chroma, (3) stuff them into the prompt, (4) call Gemini.
retrieval_qa_chain = create_retrieval_chain(retriever, combine_docs_chain)

# Turn on LangChain's verbose mode so chain steps are logged inline —
# extremely useful when demoing the pipeline in a class presentation.
set_verbose(True)


### 8e. Pretty-printing helpers

In [24]:
def wrap(s):
    """Wrap long strings to 120 columns for readable cell output."""
    return "\n".join(textwrap.wrap(s, width=120, break_long_words=False))


def formatter(result):
    """Render a retrieval-chain result as a citation-style block:
    the original query, every retrieved chunk with its provenance metadata,
    then the final grounded answer."""
    print(f"Query: {result['input']}")
    print("." * 80)

    # Walk through every retrieved chunk and emit its metadata + content.
    if "context" in result.keys():
        for idx, ref in enumerate(result["context"]):
            print("-" * 80)
            print(f"REFERENCE #{idx}")
            print("-" * 80)
            if "score" in ref.metadata:
                print(f"Matching Score: {ref.metadata['score']}")
            if "source" in ref.metadata:
                print(f"Document Source: {ref.metadata['source']}")
            if "document_name" in ref.metadata:
                print(f"Document Name: {ref.metadata['document_name']}")
            print("." * 80)
            print(f"Content: \n{wrap(ref.page_content)}")

    print("." * 80)
    print(f"Response: {wrap(result['answer'])}")
    print("." * 80)


### 8f. The `ask()` function

Public entry point for end-to-end question answering. Accepts an optional
`filters` argument that mirrors the original Colab API:

```python
filters = {
    "namespace": "document_name",
    "allow_list": ["Some-Paper.pdf", "Another-Paper.pdf"],
}
```

Internally we translate that into Chroma's native `where` filter syntax
(`{field: value}` for a single value, `{field: {"$in": [...]}}` for a list)
so callers don't need to know which vector store is underneath.


In [25]:
def ask(query, k=NUMBER_OF_RESULTS, filters=None):
    """Run a RAG query end-to-end and pretty-print the result.

    Parameters
    ----------
    query : str
        The natural-language question.
    k : int
        Number of chunks to retrieve (defaults to NUMBER_OF_RESULTS).
    filters : dict or None
        Optional metadata filter, e.g.
        ``{"namespace": "document_name", "allow_list": ["foo.pdf"]}``.
        Translated to a Chroma ``where`` clause internally.
    """
    # Per-query override of top-k.
    retriever.search_kwargs["k"] = k

    if filters:
        # Translate the original Matching-Engine-style filter into Chroma's
        # native `where` syntax.
        namespace = filters["namespace"]
        allow_list = filters["allow_list"]

        if len(allow_list) == 1:
            # Single allowed value → simple equality filter.
            chroma_filter = {namespace: allow_list[0]}
        else:
            # Multiple allowed values → use $in operator.
            chroma_filter = {namespace: {"$in": allow_list}}

        retriever.search_kwargs["filter"] = chroma_filter
    else:
        # Make sure we don't carry a stale filter over from a previous call.
        retriever.search_kwargs.pop("filter", None)

    # Invoke the chain: embed → retrieve → stuff → generate.
    result = retrieval_qa_chain.invoke({"input": query})

    # Render the result as a citation block.
    return formatter(result)


### 8g. Quick end-to-end sanity check

In [26]:
# Simplest possible end-to-end smoke test for the full RAG chain.
ask("What does epidemiology mean?")


Query: What does epidemiology mean?
................................................................................
--------------------------------------------------------------------------------
REFERENCE #0
--------------------------------------------------------------------------------
Document Source: gs://domain-docs/documents/pdfs
Document Name: Infectious-disease-epidemiology.pdf
................................................................................
Content: 
demos – people   logy – study     Epidemiology is, thus, the study of what is upon the people.  In modern terms, it is
the  science of the distribution of disease and its determinants (causes).      Epidemiology is also a process that uses
the facts at hand as clues to point to new  knowledge and solutions.  Epidemiologists have been called “disease
detectives” and
--------------------------------------------------------------------------------
REFERENCE #1
-------------------------------------------------------

## Phase 9 — Question-Answering Examples

Demonstrates two usage patterns:

1. **Filtered retrieval** — restrict the search to a specific document,
   useful when the user already knows which paper holds the answer.
2. **Open retrieval** — let the retriever pick from the entire corpus.

Each call below prints the retrieved chunks (with their source filenames)
and Gemini's grounded answer.


### 9a. Filtered query — restrict to a single document

In [27]:
# Restrict retrieval to a single PDF by exact filename match on the
# `document_name` metadata field stamped during Phase 2.
filters = {
    "namespace": "document_name",
    "allow_list": ["Infectious-disease-epidemiology.pdf"],
}

ask("What does epidemiology mean?", filters=filters)


Query: What does epidemiology mean?
................................................................................
--------------------------------------------------------------------------------
REFERENCE #0
--------------------------------------------------------------------------------
Document Source: gs://domain-docs/documents/pdfs
Document Name: Infectious-disease-epidemiology.pdf
................................................................................
Content: 
demos – people   logy – study     Epidemiology is, thus, the study of what is upon the people.  In modern terms, it is
the  science of the distribution of disease and its determinants (causes).      Epidemiology is also a process that uses
the facts at hand as clues to point to new  knowledge and solutions.  Epidemiologists have been called “disease
detectives” and
--------------------------------------------------------------------------------
REFERENCE #1
-------------------------------------------------------

### 9b. Open queries — search across the entire corpus

In [28]:
ask("What food source was linked to the 2024 large-scale norovirus outbreak across multiple schools in a Korean city?")


Query: What food source was linked to the 2024 large-scale norovirus outbreak across multiple schools in a Korean city?
................................................................................
--------------------------------------------------------------------------------
REFERENCE #0
--------------------------------------------------------------------------------
Document Source: gs://domain-docs/documents/pdfs
Document Name: A large-scale norovirus outbreak associated with kimchi consumption across multiple schools in a Korean city in 2024.pdf
................................................................................
Content: 
A large-scale norovirus outbreak associated with kimchi  consumption across multiple schools in a Korean city in 2024
Eunkyung Park, Sumi Cho, Eunyoung Kim, Jung Im Park, Ju-Hyung Lee, Jin Gwack Epidemiology and Health Volume: 47, Article
ID: e2025057 | https://doi.org/10.4178/epih.e2025057 Graphical Abstract Key Message:
------------------------

In [29]:
ask("What novel clade or sub-lineage of the monkeypox virus was associated with the 2024 outbreak in Kamituga, South Kivu province?")


Query: What novel clade or sub-lineage of the monkeypox virus was associated with the 2024 outbreak in Kamituga, South Kivu province?
................................................................................
--------------------------------------------------------------------------------
REFERENCE #0
--------------------------------------------------------------------------------
Document Source: gs://domain-docs/documents/pdfs
Document Name: Ongoing mpox outbreak in Kamituga, South Kivu province, associated with monkeypox virus of a novel Clade I sub-lineage, Democratic Republic of the Congo, 2024.pdf
................................................................................
Content: 
1 www.eurosurveillance.org Rapid communication Ongoing mpox outbreak in Kamituga, South Kivu  province, associated with
monkeypox virus of a novel  Clade I sub-lineage, Democratic Republic of the Congo,  2024 Leandre Murhula Masirika 1,2 ,
Jean Claude Udahemuka 3,4 , Leonard Schuele⁵ , Pacif

In [30]:
ask("According to the ESPAUR report for 2024 to 2025, what are the key findings regarding antimicrobial utilisation and resistance?")


Query: According to the ESPAUR report for 2024 to 2025, what are the key findings regarding antimicrobial utilisation and resistance?
................................................................................
--------------------------------------------------------------------------------
REFERENCE #0
--------------------------------------------------------------------------------
Document Source: gs://domain-docs/documents/pdfs
Document Name: ESPAUR report 2024 to 2025 - Antimicrobial utilisation and resistance.pdf
................................................................................
Content: 
English surveillance programme for antimicrobial utilisation and resistance (ESPAUR) Report 2024 to 2025  15  Data on
antimicrobial-resistant infections for the period 2019 to 2024 is presented as trends in  either numbers of patient
episodes (defined in the Annexe accompanying this report),  percentage resistance or as a rate per 100,000 population.
More detailed analysis of AM

In [31]:
ask("What are the interim estimates for the effectiveness of the COVID-19 vaccine during the 2024-2025 season?")


Query: What are the interim estimates for the effectiveness of the COVID-19 vaccine during the 2024-2025 season?
................................................................................
--------------------------------------------------------------------------------
REFERENCE #0
--------------------------------------------------------------------------------
Document Source: gs://domain-docs/documents/pdfs
Document Name: Interim Estimates of 2024–2025 COVID-19 Vaccine Effectiveness.pdf
................................................................................
Content: 
vaccines-us.html This analysis estimated 2024–2025 COVID-19 vaccine  effectiveness (VE) against COVID-19–associated
emergency  department (ED) or urgent care (UC) visits in one CDC- funded VE network and VE against COVID-19–associated
hospitalization in two CDC-funded VE networks during  September 2024–January 2025** among adults aged ≥18 years. Methods
Data Source
------------------------------------------